# Auditing components.yaml

This notebook runs the AuditSystem against the main components.yaml file to evaluate solution quality.

In [1]:
from pathlib import Path

from iacs.audit_system import (
    AuditRunner,
    RequirementCoverageAudit,
    TraceabilityAudit,
    TodoAudit,
)
from iacs.io_system import IOSystem
from iacs.registry import Registry

## Load components.yaml

In [2]:
io = IOSystem()
components_df = io.read_entity_centered_file(Path("../components/components.yaml"))
registry = Registry.from_entity_centered(components_df)

print(f"Loaded {len(components_df)} components from {components_df['entity_id'].nunique()} entities")
print(f"Component types: {registry.component_types}")

Loaded 547 components from 144 entities
Component types: ['description', 'file_info', 'id', 'input', 'output', 'parent', 'requirement', 'schema', 'system', 'todo']


## Run All Audits

In [3]:
runner = AuditRunner(audits=[
    RequirementCoverageAudit(),
    TraceabilityAudit(),
    TodoAudit(),
])

results = runner.run(registry)

print(f"All audits passed: {runner.all_passed}")
print()
for name, result in results.items():
    status = "PASS" if result.passed else "FAIL"
    print(f"{name}: {status}")

All audits passed: False

requirement_coverage: FAIL
traceability: FAIL
todo: FAIL


## Requirement Coverage Audit

Checks that all requirements have implementing solutions.

In [4]:
req_result = results["requirement_coverage"]
print(f"Passed: {req_result.passed}")
print(f"Uncovered requirements: {len(req_result.results)}")
print()

# View uncovered requirements with their descriptions using multi-component join
if not req_result.results.empty:
    uncovered_ids = req_result.results["entity_id"].tolist()
    
    # Get requirements joined with descriptions
    req_with_desc = registry.view(["requirement", "description"])
    
    # Filter to only uncovered requirements
    uncovered_details = req_with_desc[req_with_desc.index.isin(uncovered_ids)]
    
    # Show relevant columns
    display_cols = ["description.value"]
    if "requirement.value" in uncovered_details.columns:
        display_cols.append("requirement.value")
    
    print("Uncovered requirements with descriptions:")
    display(uncovered_details[display_cols])

Passed: False
Uncovered requirements: 45

Uncovered requirements with descriptions:


,description.value,requirement.value
entity_id,,
aa4b968e2d6d,We don't want to duplicate other IaC code if p...,NaN
669b4507e408,Code such as Python also provides clear defini...,NaN
4834288e3da8,Version control is essential for reliability.\n,constraint
0d2717c66a9f,A source of truth is not *a* source of truth i...,NaN
ecs_framework_base_requirement,One potential solution for allowing users to d...,NaN
f8b2f13ad59f,We need to be able to define standard parts of...,constraint
6c519747d902,Human-executed infrastructure is infrastructur...,constraint
ea91189332a0,Abstract components of infrastructure include ...,constraint
2d994b129844,We of course want the data format to be compat...,"{'value': 'quality', 'priority': 0.1}"


## Traceability Audit

Checks that all entities trace back to requirements.

In [5]:
trace_result = results["traceability"]
print(f"Passed: {trace_result.passed}")
print(f"Orphan entities: {len(trace_result.results)}")
print()
if not trace_result.results.empty:
    print("Entities not tracing to requirements:")
    display(trace_result.results)

Passed: False
Orphan entities: 66

Entities not tracing to requirements:


,entity_id
0,080505fba0c9
26,76136e1235d8
27,6c837dbd9816
35,describe_hierarchical_systems
43,ac8ff4d06250
...,...
139,acab60c10335
140,8f1457ff8841
141,3478afd59783
142,7e1466700007


## Todo Audit

Reports outstanding todos.

In [6]:
todo_result = results["todo"]
print(f"Passed: {todo_result.passed}")
print(f"Entities with todos: {len(todo_result.results)}")
print()
if todo_result.messages:
    print("Outstanding todos:")
    display(todo_result.results)

Passed: False
Entities with todos: 7

Outstanding todos:


,entity_id
0,a1d2e01e0c6b
1,3fff1d444644
2,f7a5bd87edc7
3,38a498d554d8
4,06123ab1c933
5,categorical
6,8f2861483c06
